# Customer insights 2023 (fixed)

Same data, same eleven questions, honest analysis. Each *Fix* note names the fallacy.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score

pd.set_option("display.width", 120)
rng = np.random.default_rng(0)

meters = pd.read_csv("../../data/meters.csv", parse_dates=["signup_date"])
meters["region"] = meters["region"].str.title()
readings = pd.read_csv("../../data/meter_readings_daily.csv", parse_dates=["date"])
readings = readings[readings["meter_id"].isin(meters["meter_id"])]
readings["month"] = readings["date"].dt.month
annual = readings.groupby("meter_id")["kwh"].sum().rename("kwh_2023")
cust = meters.merge(annual, on="meter_id", how="inner")
res = cust[cust["customer_type"] == "residential"]

*Fix 1 — Simpson's paradox.* Regional means mix residential and SME customers; SMEs use
~8x more, and Scotland has 5% SMEs vs 12-16% elsewhere. Compare within customer type.

In [2]:
strat = cust.groupby(["region", "customer_type"])["kwh_2023"].mean().unstack().round(0)
strat["sme_share"] = cust.groupby("region")["customer_type"].apply(lambda x: (x == "sme").mean()).round(3)
strat["res_vs_wales"] = (strat["residential"] / strat.loc["Wales", "residential"] - 1).round(3)
strat

customer_type,residential,sme,sme_share,res_vs_wales
region,,,,
London,3351.0,24607.0,0.121,0.023
Midlands,3216.0,23480.0,0.156,-0.019
North,2979.0,27520.0,0.106,-0.091
Scotland,3021.0,25066.0,0.053,-0.078
Wales,3277.0,24093.0,0.152,0.000


*Fix 2 — regression to the mean.* Selecting the top 50 on one noisy day guarantees they look
lower on any other day; no behaviour changed. Controls: the bottom 50 rise by a similar
amount, and selecting on the *later* day makes the same meters look like they went *up*.

In [3]:
wide = readings[readings["meter_id"].isin(res["meter_id"])].pivot(index="meter_id", columns="date", values="kwh")
d0, d1 = pd.Timestamp("2023-01-16"), pd.Timestamp("2023-01-23")
def chg(ids, a, b): return wide.loc[ids, b].mean() / wide.loc[ids, a].mean() - 1
top, bot = wide[d0].nlargest(50).index, wide[d0].nsmallest(50).index
top_later = wide[d1].nlargest(50).index
print(f"top-50 selected on d0:     {chg(top, d0, d1):+.1%}  (d0 -> d1)")
print(f"bottom-50 selected on d0:  {chg(bot, d0, d1):+.1%}")
print(f"top-50 selected on d1, looking back: {chg(top_later, d1, d0):+.1%}  (d1 -> d0)")
print(f"same meters, 30-day averages either side: {wide.loc[top, d0 + pd.Timedelta(days=1):d0 + pd.Timedelta(days=30)].mean().mean() / wide.loc[top, d0 - pd.Timedelta(days=30):d0 - pd.Timedelta(days=1)].mean().mean() - 1:+.1%}")

top-50 selected on d0:     -19.0%  (d0 -> d1)
bottom-50 selected on d0:  +17.7%
top-50 selected on d1, looking back: -21.6%  (d1 -> d0)
same meters, 30-day averages either side: +0.2%


*Fix 3 — survivorship / seasonality.* Two things: (a) H2 < H1 is the heating season, not a
trend (compare against a seasonal baseline or the same half of another year); (b) filtering to
"complete" meters is harmless here because gaps are random, but if gaps were caused by churn
or by faults on high users the filter would bias the answer. Simulation below.

In [4]:
days = readings.groupby("meter_id").size()
half = readings.assign(half=np.where(readings["month"] <= 6, "H1", "H2"))
growth = half.groupby(["meter_id", "half"])["kwh"].mean().unstack()
growth["g"] = growth["H2"] / growth["H1"] - 1
print(f"H2/H1 - 1: all meters {growth['g'].mean():+.1%}; complete meters {growth.loc[days[days >= 360].index, 'g'].mean():+.1%}")
# seasonal expectation from the residential monthly profile: H2 vs H1 with no trend at all
prof = readings.groupby("month")["kwh"].mean()
print(f"seasonal H2/H1 - 1 from the monthly profile: {prof.loc[7:12].mean() / prof.loc[1:6].mean() - 1:+.1%}")
# simulate churn-correlated gaps: meters whose H2 usage fell most drop out of 'complete'
churn = growth["g"].nsmallest(60).index                      # 60 fastest-falling meters leave mid-year
kept = growth.drop(churn)
print(f"if the 60 fastest-falling meters churned and were excluded: {kept['g'].mean():+.1%}  (bias of {kept['g'].mean() - growth['g'].mean():+.1%})")

H2/H1 - 1: all meters -11.2%; complete meters -10.8%
seasonal H2/H1 - 1 from the monthly profile: -10.8%
if the 60 fastest-falling meters churned and were excluded: -10.2%  (bias of +0.9%)


*Fix 4 — pseudo-replication and multiple comparisons.* Twelve monthly tests on the *same*
meters are one finding measured twelve times, not twelve confirmations. Test once at meter
level (annual mean per meter) and correct for the five regions tested (Benjamini-Hochberg).

In [5]:
per_meter = res[["meter_id", "region", "kwh_2023"]]
rows = []
for reg in sorted(per_meter["region"].unique()):
    a = per_meter.loc[per_meter["region"] == reg, "kwh_2023"]; b = per_meter.loc[per_meter["region"] != reg, "kwh_2023"]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    rows.append({"region": reg, "n": len(a), "diff_pct": a.mean() / b.mean() - 1, "ci_low": (a.mean() - b.mean() - 1.96 * se) / b.mean(), "ci_high": (a.mean() - b.mean() + 1.96 * se) / b.mean(), "p": p})
t5 = pd.DataFrame(rows).sort_values("p")
m_ = len(t5); t5["p_bh"] = np.minimum.accumulate((t5["p"] * m_ / np.arange(1, m_ + 1)).to_numpy()[::-1])[::-1]
t5.round(3)

,region,n,diff_pct,ci_low,ci_high,p,p_bh
0,London,87,0.085,0.015,0.155,0.018,0.092
2,North,59,-0.078,-0.153,-0.003,0.045,0.114
3,Scotland,54,-0.060,-0.140,0.020,0.146,0.244
4,Wales,28,0.036,-0.072,0.145,0.517,0.646
1,Midlands,38,0.015,-0.065,0.095,0.709,0.709


*Fix 5 — base rate.* 88.7% of meters have no solar, so "always no" scores 88.7%. Report
recall/AUC for the positive class instead of accuracy.

In [6]:
X = pd.get_dummies(cust[["region", "tariff", "customer_type"]].fillna("unknown"), drop_first=True).astype(float)
X["kwh_2023"] = cust["kwh_2023"]
y = cust["has_solar"].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
print(f"accuracy {clf.score(X_te, y_te):.3f} | always-no accuracy {1 - y_te.mean():.3f} | AUC {roc_auc_score(y_te, proba):.3f} | recall of solar at 0.5 threshold {recall_score(y_te, proba > 0.5):.2f}")

accuracy 0.889 | always-no accuracy 0.889 | AUC 0.511 | recall of solar at 0.5 threshold 0.00


*Fix 6 — denominator neglect.* London has the most solar customers because it has the most
customers. The *rate* is the same in London, North and Scotland; Midlands is the outlier.

In [7]:
meters.groupby("region")["has_solar"].agg(count="sum", meters="size", rate="mean").round(3).sort_values("rate", ascending=False)

,count,meters,rate
region,,,
Scotland,8,57,0.140
North,9,66,0.136
London,13,99,0.131
Wales,3,33,0.091
Midlands,1,45,0.022


*Fix 7 — self-selection and pseudo-replication.* Customers choose TOU; the comparison is
observational. And 95k daily rows are not 95k independent observations: the unit is the
meter (46 vs 220). At meter level the difference is not distinguishable from zero.

In [8]:
rr = readings.merge(res[["meter_id", "tariff"]], on="meter_id")
pm = rr.groupby(["meter_id", "tariff"])["kwh"].mean().reset_index()
a, b = pm.loc[pm["tariff"] == "TOU", "kwh"], pm.loc[pm["tariff"] != "TOU", "kwh"]
t, p = stats.ttest_ind(a, b, equal_var=False)
se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
print(f"TOU vs others, meter level: diff {a.mean() / b.mean() - 1:+.1%}, 95% CI [{(a.mean() - b.mean() - 1.96 * se) / b.mean():+.1%}, {(a.mean() - b.mean() + 1.96 * se) / b.mean():+.1%}], p = {p:.2f}  (n = {len(a)} vs {len(b)} meters)")
print("daily-row test for comparison: p =", f"{stats.ttest_ind(rr.loc[rr['tariff'] == 'TOU', 'kwh'], rr.loc[rr['tariff'] != 'TOU', 'kwh']).pvalue:.0e}")

TOU vs others, meter level: diff +4.9%, 95% CI [-3.5%, +13.3%], p = 0.25  (n = 46 vs 209 meters)
daily-row test for comparison: p = 1e-35


*Fix 8 — ecological fallacy.* A correlation across 5 regional averages says nothing about
individuals. At meter level solar customers use the same as everyone else (the panels offset
a small summer amount in this data).

In [9]:
g = res.groupby("has_solar")["kwh_2023"].agg(["mean", "size"]).round(0)
a, b = res.loc[res["has_solar"], "kwh_2023"], res.loc[~res["has_solar"], "kwh_2023"]
print(g)
print(f"meter-level diff {a.mean() / b.mean() - 1:+.1%}, p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.2f}; the regional correlation was computed on n = 5 points")

             mean  size
has_solar              
False      3169.0   233
True       3212.0    33
meter-level diff +1.4%, p = 0.78; the regional correlation was computed on n = 5 points


*Fix 9 — extrapolating a straight line through a seasonal cycle.* Jan-Jun is the falling
half of an annual cycle. Fit the cycle (a cosine, or simply use last year's profile) instead.

In [10]:
mt = readings.groupby("month")["kwh"].sum()
months = np.arange(1, 13)
def seasonal(mo, a, b, c): return a + b * np.cos(2 * np.pi * (mo - c) / 12)
from scipy.optimize import curve_fit
(a, b, c), _ = curve_fit(seasonal, np.arange(1, 7), mt.loc[1:6].values, p0=[140_000, 50_000, 1])
q4_fit = seasonal(np.array([10, 11, 12]), a, b, c).sum()
slope, intercept = np.polyfit(np.arange(1, 7), mt.loc[1:6].values, 1)
q4_lin = np.clip(np.polyval([slope, intercept], [10, 11, 12]), 0, None).sum()
print(f"Q4 actual {mt.loc[10:12].sum() / 1e3:,.0f} MWh | seasonal fit on Jan-Jun {q4_fit / 1e3:,.0f} MWh | linear extrapolation {q4_lin / 1e3:,.0f} MWh")

Q4 actual 495 MWh | seasonal fit on Jan-Jun 481 MWh | linear extrapolation 32 MWh


*Fix 10 — cherry-picked window.* January-to-July is the peak-to-trough of the heating
season. Compare like with like: the full-year profile, or the same month year on year.

In [11]:
print((mt / 1e3).round(0).to_dict())
print(f"Jan -> Jul {mt[7] / mt[1] - 1:+.0%}; Jul -> Dec {mt[12] / mt[7] - 1:+.0%}; Jan -> Dec {mt[12] / mt[1] - 1:+.1%}")

{1: 193.0, 2: 169.0, 3: 170.0, 4: 141.0, 5: 119.0, 6: 96.0, 7: 93.0, 8: 100.0, 9: 116.0, 10: 144.0, 11: 164.0, 12: 187.0}
Jan -> Jul -52%; Jul -> Dec +100%; Jan -> Dec -3.2%


*Fix 11 — significance is not size, and rows are not customers.* 5% is a small effect; at the
meter level (127 vs 82 meters) it is not significant. Report effect size with a CI at the right
unit of analysis.

In [12]:
a, b = pm.loc[pm["tariff"] == "Fixed", "kwh"], pm.loc[pm["tariff"] == "Variable", "kwh"]
t, p = stats.ttest_ind(a, b, equal_var=False)
se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
print(f"Variable vs Fixed, meter level: diff {b.mean() / a.mean() - 1:+.1%}, 95% CI [{(b.mean() - a.mean() - 1.96 * se) / a.mean():+.1%}, {(b.mean() - a.mean() + 1.96 * se) / a.mean():+.1%}], p = {p:.2f}  (n = {len(a)} vs {len(b)})")

Variable vs Fixed, meter level: diff +5.4%, 95% CI [-2.2%, +13.1%], p = 0.17  (n = 127 vs 82)


## Results (honest)

1. Within customer type, regions differ by at most ~10%; the 35% gap was SME mix.
2. Top users "self-correcting" is regression to the mean; the bottom 50 "rose" by the same logic.
3. H2 < H1 is the heating season; the seasonal profile predicts it. No trend.
4. One finding, tested once: London residential is +8.5% (CI +1.5% to +15.5%), p = 0.018 raw but 0.09 after BH across five regions: suggestive, not established. North does not survive either.
5. The solar model is worse than "always no" would look; AUC ~0.5, recall ~0.
6. Solar *rate* is flat across London / North / Scotland; Midlands is the outlier.
7. TOU: observational, and not significant at meter level.
8. Solar customers use the same as others at meter level.
9. Q4 ~ 500 MWh, close to Q1; the linear extrapolation is off by a factor of 15.
10. Jan-Jul -52% is seasonality; Jan-Dec is -3%.
11. Variable vs Fixed: +5% ± ~7% at meter level; not established.